# PID Controllers

A **feedback controller** measures the gap between where the system *is* and where you *want* it, generating a corrective signal $u(t)$ to close that gap.

The **tracking error** is defined as:
$$e(t) = r(t) - y(t)$$
where $r(t)$ is the reference setpoint and $y(t)$ is the plant output.

The **PID control law** combines three complementary actions on $e(t)$:
$$\boxed{u(t) = \underbrace{K_p\,e(t)}_{\text{Proportional}} + \underbrace{K_i\!\int_0^t\! e(\tau)\,d\tau}_{\text{Integral}} + \underbrace{K_d\,\dot{e}(t)}_{\text{Derivative}}}$$

In the Laplace domain the controller transfer function is:
$$C(s) = K_p + \frac{K_i}{s} + K_d\,s$$

| Term | Responds to | Key effect |
|------|------------|------------|
| $K_p\,e$ | **Present** error | Immediate correction — high $K_p$ risks oscillation |
| $K_i\int e\,d\tau$ | **Accumulated past** error | Eliminates steady-state error — can destabilize |
| $K_d\,\dot{e}$ | **Future trend** (error rate) | Damps overshoot — amplifies measurement noise |

The D term acts as **anticipatory braking**: when the error is shrinking quickly, $\dot{e}<0$, so D subtracts from $u(t)$, preventing the system from overshooting.

The interactive below isolates each contribution for a representative decaying error $e(t)=e^{-0.4t}$.  Raise each gain and observe how the shape of $u(t)$ changes.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider

def plot_pid_components(Kp, Ki, Kd):
    t  = np.linspace(0, 10, 1000)
    dt = t[1] - t[0]
    e  = np.exp(-0.4 * t)          # decaying error: error closing toward zero

    P = Kp * e
    I = Ki * np.cumsum(e) * dt     # integral accumulates history of error
    D = Kd * np.gradient(e, dt)    # derivative is negative: e is decreasing
    u = P + I + D

    fig, axes = plt.subplots(1, 5, figsize=(16, 4))
    fig.suptitle(
        f'PID components on e(t)=exp(−0.4t)  |  Kp={Kp:.1f}, Ki={Ki:.1f}, Kd={Kd:.2f}',
        fontsize=11
    )
    for ax, (sig, label, color) in zip(axes, [
        (e,  'e(t) — error',                '#546E7A'),
        (P,  'Kp·e — P term',          '#1565C0'),
        (I,  'Ki·∫e·dt — I term', '#2E7D32'),
        (D,  'Kd·de/dt — D term',      '#E65100'),
        (u,  'u = P + I + D — total',       '#6A1B9A'),
    ]):
        ax.plot(t, sig, color=color, lw=2)
        ax.set_title(label, fontsize=9)
        ax.set_xlabel('Time (s)')
        ax.axhline(0, color='k', lw=0.6)
        ax.grid(True, alpha=0.25)
    axes[0].set_ylabel('Amplitude')
    plt.tight_layout()
    plt.show()

interactive(
    plot_pid_components,
    Kp=FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description='Kp', continuous_update=False),
    Ki=FloatSlider(value=0.5, min=0.0, max=5.0, step=0.1, description='Ki', continuous_update=False),
    Kd=FloatSlider(value=0.3, min=0.0, max=3.0, step=0.1, description='Kd', continuous_update=False),
)


interactive(children=(FloatSlider(value=1.0, continuous_update=False, description='Kp', max=5.0), FloatSlider(…

## Closed-Loop System

We place the PID in a unity-feedback loop around a **second-order plant**:
$$G(s) = \frac{1}{(s+1)^2}$$

With $C(s) = \frac{K_d s^2 + K_p s + K_i}{s}$ the closed-loop transfer function is:
$$T(s) = \frac{C(s)\,G(s)}{1 + C(s)\,G(s)} = \frac{K_d\,s^2 + K_p\,s + K_i}{s^3 + (2+K_d)\,s^2 + (1+K_p)\,s + K_i}$$

**Stability** (Routh–Hurwitz) requires all three conditions:
$$K_i > 0, \qquad K_p > -1, \qquad \underbrace{(2+K_d)(1+K_p) > K_i}_{\text{cross-condition}}$$

The cross-condition is the one students usually violate: pushing $K_i$ too high relative to the damping provided by $K_d$ and $K_p$ drives the closed-loop poles into the right half-plane.

**Steady-state error** with proportional control only ($K_i = 0$):
$$e_{ss} = \frac{1}{1 + K_p} > 0$$
Adding $K_i > 0$ makes the controller Type-1, forcing $e_{ss} \to 0$ for any step reference.

### Performance metrics

| Metric | Symbol | Definition |
|--------|--------|-----------|
| Rise time | $t_r$ | Time from 10 % to 90 % of $y_{ss}$ |
| Overshoot | $M_p$ | $(y_\text{max} - y_{ss}) / y_{ss} \times 100\,\%$ |
| Settling time | $t_s$ | Last time $\lvert y - y_{ss}\rvert > 0.02\,y_{ss}$ |
| ISE | $\int_0^\infty e^2\,dt$ | Total squared tracking error — lower is better |

Use the sliders to find gains that minimise ISE while keeping $M_p$ and $t_s$ acceptable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lti, step
from ipywidgets import interactive, FloatSlider

def build_cl(Kp, Ki, Kd):
    if Ki == 0:
        # s cancels from num and den; use reduced 2nd-order form
        return lti([Kd, Kp] if Kd > 0 else [Kp], [1, 2 + Kd, 1 + Kp])
    return lti([Kd, Kp, Ki], [1, 2 + Kd, 1 + Kp, Ki])

def compute_metrics(t, y, r=1.0):
    y_ss = float(y[-1])
    e    = r - y
    ise  = float(np.trapezoid(e**2, t))
    Mp   = (float(np.max(y)) - y_ss) / y_ss * 100 if y_ss > 1e-6 else float('nan')
    i10  = np.searchsorted(y, 0.10 * y_ss)
    i90  = np.searchsorted(y, 0.90 * y_ss)
    tr   = float(t[min(i90, len(t)-1)] - t[min(i10, len(t)-1)])
    out  = np.where(np.abs(y - y_ss) > 0.02 * abs(y_ss))[0]
    ts   = float(t[out[-1]]) if len(out) else 0.0
    return dict(y_ss=y_ss, Mp=Mp, tr=tr, ts=ts, ess=r - y_ss, ise=ise)

def plot_step_response(Kp, Ki, Kd):
    t = np.linspace(0, 60, 4000)
    _, y = step(build_cl(Kp, Ki, Kd), T=t)

    if np.max(np.abs(y)) > 200:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.text(0.5, 0.5, 'UNSTABLE\nIncrease Kd or reduce Ki',
                ha='center', va='center', fontsize=16, color='#C62828',
                transform=ax.transAxes)
        ax.set_title(f'Kp={Kp:.1f}  Ki={Ki:.1f}  Kd={Kd:.2f}')
        ax.axis('off')
        plt.show()
        return

    m = compute_metrics(t, y)

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(t, y, '#1565C0', lw=2, label='Output y(t)')
    ax.axhline(1.0,      color='gray',    ls='--', lw=1.2, label='Reference r = 1')
    ax.axhline(m['y_ss'], color='#388E3C', ls=':',  lw=1.2,
               label=f"Steady state = {m['y_ss']:.3f}")
    ax.axhspan(m['y_ss']*0.98, m['y_ss']*1.02,
               alpha=0.10, color='#388E3C', label='±2 % band')

    info = (
        f"Rise time      {m['tr']:.2f} s\n"
        f"Overshoot      {m['Mp']:.1f} %\n"
        f"Settling time  {m['ts']:.2f} s\n"
        f"SSE            {m['ess']:.4f}\n"
        f"ISE            {m['ise']:.3f}"
    )
    ax.text(0.98, 0.97, info, transform=ax.transAxes, fontsize=10,
            ha='right', va='top', family='monospace',
            bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.88))

    ax.set(xlabel='Time (s)', ylabel='Output',
           title=f'Closed-Loop Step Response  |  Kp={Kp:.1f}, Ki={Ki:.1f}, Kd={Kd:.2f}',
           ylim=(-0.05, max(1.65, float(np.max(y)) + 0.1)))
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

interactive(
    plot_step_response,
    Kp=FloatSlider(value=2.0, min=0.0, max=10.0, step=0.1, description='Kp', continuous_update=False),
    Ki=FloatSlider(value=1.0, min=0.0, max=10.0, step=0.1, description='Ki', continuous_update=False),
    Kd=FloatSlider(value=1.0, min=0.0, max=10.0, step=0.1, description='Kd', continuous_update=False),
)


interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='Kp', max=10.0), FloatSlider…

## Gain Effects and Pole Placement

The denominator of $T(s)$ is the **characteristic polynomial**:
$$\Delta(s) = s^3 + (2+K_d)\,s^2 + (1+K_p)\,s + K_i$$

Its three roots — the **closed-loop poles** — fully determine the transient response.  Poles in the **left half-plane** (Re$\,<0$) produce stable, decaying modes; poles near the imaginary axis give slow oscillations; poles in the right half-plane give a diverging response.

| Gain | Primary effect on poles | Caution |
|------|------------------------|---------|
| $\uparrow K_p$ | Pushes poles away from origin → faster response | Too fast → poles approach Im axis → overshoot |
| $\uparrow K_i$ | Bends poles inward to ensure zero SSE | Violates cross-condition → instability |
| $\uparrow K_d$ | Increases $s^2$ coefficient → poles move left → more damping | Noise amplification at high frequency |

**Reading the pole map:** the further left from the imaginary axis, the faster that mode decays.  A complex conjugate pair $\sigma \pm j\omega$ produces an oscillation at frequency $\omega$ that decays at rate $\sigma$.

**Suggested experiments:**

1. Start with $K_p=2$, $K_i=0$, $K_d=0$ — observe steady-state error  
2. Add $K_i=1$ — SSE vanishes; note the change in pole positions  
3. Increase $K_i$ past $(2+K_d)(1+K_p)$ — watch poles enter the RHP and the system go unstable  
4. Raise $K_d$ — poles move left, damping increases

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import lti, step
from ipywidgets import interactive, FloatSlider

def build_cl(Kp, Ki, Kd):
    if Ki == 0:
        return lti([Kd, Kp] if Kd > 0 else [Kp], [1, 2 + Kd, 1 + Kp])
    return lti([Kd, Kp, Ki], [1, 2 + Kd, 1 + Kp, Ki])

def plot_poles_and_response(Kp, Ki, Kd):
    den   = [1, 2 + Kd, 1 + Kp, Ki] if Ki > 0 else [1, 2 + Kd, 1 + Kp]
    poles = np.roots(den)
    stable = all(p.real < 0 for p in poles)

    t = np.linspace(0, 60, 4000)
    diverged = not stable
    if stable:
        _, y = step(build_cl(Kp, Ki, Kd), T=t)
        diverged = float(np.max(np.abs(y))) > 200

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(
        f'Kp={Kp:.1f}  Ki={Ki:.1f}  Kd={Kd:.2f}   '
        f'(stability bound Ki < {(2+Kd)*(1+Kp):.2f})',
        fontsize=12
    )

    # ── Pole-zero map ──────────────────────────────────────────────────────
    ax1.axvline(0, color='k', lw=0.8)
    ax1.axhline(0, color='k', lw=0.8)
    ax1.axvspan(-12, 0, alpha=0.06, color='#388E3C', label='Stable region (Re < 0)')
    for p in poles:
        c = '#D32F2F' if p.real >= 0 else '#1565C0'
        ax1.scatter(p.real, p.imag, marker='x', s=180, color=c, lw=2.5, zorder=5)
        ax1.annotate(
            f'  {p.real:.2f}{p.imag:+.2f}j',
            (p.real, p.imag), fontsize=9, color=c
        )
    ax1.set(xlim=(-8, 2), ylim=(-6, 6),
            xlabel='Re(s)', ylabel='Im(s)',
            title='Closed-Loop Poles  (blue stable, red unstable)')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.25)

    # ── Step response ──────────────────────────────────────────────────────
    if not diverged:
        ax2.plot(t, y, '#1565C0', lw=2)
        ax2.axhline(1.0, color='gray', ls='--', lw=1.2, label='Reference')
        ax2.axhline(float(y[-1]), color='#388E3C', ls=':', lw=1,
                    label=f'y_ss = {float(y[-1]):.3f}')
        ax2.set_ylim(-0.05, max(1.65, float(np.max(y)) + 0.1))
        ax2.legend(fontsize=10)
    else:
        ax2.text(0.5, 0.5, 'UNSTABLE', ha='center', va='center',
                 fontsize=24, color='#C62828', transform=ax2.transAxes,
                 fontweight='bold')
    ax2.set(xlabel='Time (s)', ylabel='Output', title='Step Response')
    ax2.grid(True, alpha=0.25)

    plt.tight_layout()
    plt.show()

interactive(
    plot_poles_and_response,
    Kp=FloatSlider(value=2.0, min=0.0, max=10.0, step=0.1, description='Kp', continuous_update=False),
    Ki=FloatSlider(value=1.0, min=0.0, max=10.0, step=0.1, description='Ki', continuous_update=False),
    Kd=FloatSlider(value=1.0, min=0.0, max=10.0, step=0.1, description='Kd', continuous_update=False),
)


interactive(children=(FloatSlider(value=2.0, continuous_update=False, description='Kp', max=10.0), FloatSlider…